# Distance Calculations — Grids & Hexbins

Unified, parameterized replacement for `00e_` (hexbins) and `00g_` (grids).

**All steps are idempotent.** Re-run any section at any time without recomputing
existing columns. To add a new distance variable:
1. Make sure the source feature table exists in PostGIS (or add a load call in Section 2).
2. Append a row to `DISTANCE_SOURCES` in the Parameters cell.
3. Re-run **Section 4** only.

| Section | Contents |
|---|---|
| 1 | Parameters, imports, paths, DB connection, helpers |
| 2 | Reference feature data (roads, markets, rivers, pop centers, settlements) |
| 3 | Geometry upload (grids / hexbins) |
| 4 | Planar distance calculations |
| 5 | Geodesic distances + planar/geodesic diff columns (grids only) |
| 6 | Settlement distances (two-pass) |
| 7 | Export to GeoJSON |

## Section 1 — Setup

In [ ]:
# ============================================================
# PARAMETERS — edit only this cell to control what runs
# ============================================================

AREAS = [250, 62, 15]

# PostGIS target table names
GRID_TABLES = {250: "grid_250k_v3", 62: "grid_62k_v3", 15: "grid_15k_v3"}
HEX_TABLES  = {250: "hex_250k",     62: "hex_62k",     15: "hex_15k"}

# Source GeoJSON filenames (looked up in data_dir / processed)
GRID_GEOJSON = {
    250: "uganda_grids_250k_lcluc_10k_buffer_v3.geojson",
    62:  "uganda_grids_62k_lcluc_10k_buffer_v3.geojson",
    15:  "uganda_grids_15k_lcluc_10k_buffer_v3.geojson",
}
HEX_GEOJSON = {
    250: "uganda_hexbins_250k_lcluc_10K_buffer_v2.geojson",
    62:  "uganda_hexbins_62k_lcluc_10K_buffer_v2.geojson",
    15:  "uganda_hexbins_15k_lcluc_10K_buffer_v2.geojson",
}

# Which geometry types to include in all steps
PROCESS_GRIDS   = True
PROCESS_HEXBINS = True

# Force re-upload to PostGIS even if the table already exists.
# Set True only when the underlying source file has changed.
FORCE_RELOAD_GEOMETRY = False  # grids / hexbins
FORCE_RELOAD_FEATURES = False  # roads, markets, rivers, pop centers, settlements

# Compute geodesic distances and planar/geodesic diff columns.
# Only applies to grids; hexbins are always skipped.
COMPUTE_GEODESIC = True

# Write processed tables back to GeoJSON after all calculations.
EXPORT_RESULTS = True

In [ ]:
# ============================================================
# DISTANCE SOURCES
#
# Each row: (output_column_name, postgis_feature_table, applies_to)
#   applies_to: "both" | "grids" | "hexbins"
#
# To add a new distance variable:
#   1. Ensure the feature table is loaded in Section 2.
#   2. Append a row here.
#   3. Re-run the distance loop cell in Section 4 only.
# ============================================================

DISTANCE_SOURCES = [
    # (output_col,                     postgis_table,          applies_to)
    ("dist_to_road",                 "roads",                "both"),
    ("dist_to_market",               "markets",              "both"),
    ("dist_to_rivers",               "rivers",               "both"),
    ("dist_to_rivers_and_streams",   "rivers_and_streams",   "both"),
    ("dist_to_rivers_plus",          "rivers_plus",          "both"),
    ("dist_to_pop_center_1",         "pop_centers_1",        "both"),
    ("dist_to_pop_center_2",         "pop_centers_2",        "both"),
    ("dist_to_pop_center_3",         "pop_centers_3",        "both"),
    ("dist_to_pop_center_4",         "pop_centers_4",        "hexbins"),
]

In [ ]:
import os
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.validation import make_valid
import yaml
import psycopg2
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [ ]:
def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "paths.yaml").exists():
            return candidate
    raise FileNotFoundError("configs/paths.yaml not found")

project_root = find_project_root()

with open(project_root / "configs" / "paths.yaml") as f:
    paths = yaml.safe_load(f)

data_dir      = project_root / paths["data"]["processed"]
data_external = project_root / paths["data"]["external"]
maps_dir      = project_root / paths["outputs"]["dynamic_maps"]

In [ ]:
load_dotenv()

conn = psycopg2.connect(
    dbname=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
)

engine = create_engine(
    f"postgresql://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

In [ ]:
# ---- SQL helpers ----

def run_sql(sql_str: str):
    """Execute one or more SQL statements and commit."""
    with conn.cursor() as cur:
        try:
            cur.execute(sql_str)
            conn.commit()
        except Exception as e:
            conn.rollback()
            print(f"SQL error: {e}")
            raise


def table_exists(table_name: str, schema: str = "public") -> bool:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT EXISTS (
                SELECT 1 FROM information_schema.tables
                WHERE table_schema = %s AND table_name = %s
            )
        """, (schema, table_name))
        return cur.fetchone()[0]


# ---- Upload helpers ----

def upload_if_missing(gdf: gpd.GeoDataFrame, table_name: str, force: bool = False):
    """Upload a GeoDataFrame to PostGIS only when the table is absent or force=True."""
    if not force and table_exists(table_name):
        print(f"  {table_name}: already in PostGIS — skipping "
              f"(set FORCE_RELOAD_FEATURES=True to overwrite)")
        return
    print(f"  {table_name}: uploading {len(gdf):,} rows ...")
    gdf.to_postgis(table_name, engine, if_exists="replace", index=False)
    print(f"  {table_name}: done")


# ---- Index / centroid helpers ----

def ensure_spatial_index(table_name: str, geom_col: str = "geometry"):
    run_sql(f"""
        CREATE INDEX IF NOT EXISTS {table_name}_{geom_col}_gist
        ON {table_name} USING GIST ({geom_col});
    """)


def ensure_centroid_column(table_name: str):
    """Add and populate a projected centroid column; create a spatial index on it."""
    run_sql(f"""
        ALTER TABLE {table_name}
            ADD COLUMN IF NOT EXISTS centroid geometry(Point, 32636);
        UPDATE {table_name}
            SET centroid = ST_Centroid(geometry)
        WHERE centroid IS NULL;
    """)
    run_sql(f"""
        CREATE INDEX IF NOT EXISTS {table_name}_centroid_gist
        ON {table_name} USING GIST (centroid);
    """)

In [ ]:
# Derive the active geometry tables list once; reused in every section.
active_tables = []
if PROCESS_GRIDS:
    active_tables += [("grids",   area, GRID_TABLES[area]) for area in AREAS]
if PROCESS_HEXBINS:
    active_tables += [("hexbins", area, HEX_TABLES[area])  for area in AREAS]

print("Active tables:")
for geom_type, area, tbl in active_tables:
    print(f"  [{geom_type:7}] {tbl}")

## Section 2 — Reference Feature Data

Loads roads, markets, rivers, population centers, and settlements into PostGIS.
Each upload is skipped when the table already exists (`FORCE_RELOAD_FEATURES=False`).

**To add a new feature layer:** load it here and call `upload_if_missing`, then add
the corresponding row to `DISTANCE_SOURCES` above.

In [ ]:
# ----- Roads & markets (pre-processed files) -----
roads   = gpd.read_file(data_dir / "uganda_refugee_regions_osm_roads_clean_v1.geojson").to_crs(32636)
markets = gpd.read_file(data_dir / "uganda_markets_clean_v1.geojson").to_crs(32636)

# ----- Rivers (HOTOSM waterways, filtered by type) -----
_waterways = gpd.read_file(data_external / "hotosm_waterways.geojson").to_crs(32636)

rivers             = _waterways[_waterways["waterway"].isin(["river"])]
rivers_and_streams = _waterways[_waterways["waterway"].isin(["river", "stream"])]
rivers_plus        = _waterways[_waterways["waterway"].isin(
    ["river", "stream", "canal", "wadi", "brook", "waterfall", "rapids", "oxbow"]
)]

# ----- Population centres (HOTOSM, filtered by place tag) -----
_pop = gpd.read_file(data_external / "hotosm_populated_places.geojson").to_crs(32636)

pop_centers_1 = _pop[_pop["place"].isin(["city", "town", "village", "hamlet"])]
pop_centers_2 = _pop[_pop["place"].isin(["city", "town", "village"])]
pop_centers_3 = _pop[_pop["place"].isin(["city", "town"])]
pop_centers_4 = _pop[_pop["place"].isin(["city"])]

# ----- Settlement boundaries -----
settlements = gpd.read_file(
    data_dir / "UNHCR_poc_boundaries-Uganda_attributed_deduped.geojson"
).to_crs(32636)

In [ ]:
print("Uploading reference feature tables ...")
upload_if_missing(roads,              "roads",              force=FORCE_RELOAD_FEATURES)
upload_if_missing(markets,            "markets",            force=FORCE_RELOAD_FEATURES)
upload_if_missing(rivers,             "rivers",             force=FORCE_RELOAD_FEATURES)
upload_if_missing(rivers_and_streams, "rivers_and_streams", force=FORCE_RELOAD_FEATURES)
upload_if_missing(rivers_plus,        "rivers_plus",        force=FORCE_RELOAD_FEATURES)
upload_if_missing(pop_centers_1,      "pop_centers_1",      force=FORCE_RELOAD_FEATURES)
upload_if_missing(pop_centers_2,      "pop_centers_2",      force=FORCE_RELOAD_FEATURES)
upload_if_missing(pop_centers_3,      "pop_centers_3",      force=FORCE_RELOAD_FEATURES)
upload_if_missing(pop_centers_4,      "pop_centers_4",      force=FORCE_RELOAD_FEATURES)
upload_if_missing(settlements,        "settlements",        force=FORCE_RELOAD_FEATURES)
print("Feature upload complete.")

In [ ]:
# Spatial indexes on all feature tables
for feat_table in [
    "roads", "markets",
    "rivers", "rivers_and_streams", "rivers_plus",
    "pop_centers_1", "pop_centers_2", "pop_centers_3", "pop_centers_4",
    "settlements",
]:
    ensure_spatial_index(feat_table)
print("Feature indexes ok.")

## Section 3 — Geometry Upload

Loads grid / hexbin GeoJSONs into PostGIS and adds centroid columns.

In [ ]:
import geoalchemy2  # required for to_postgis with geometry types

for geom_type, area, table_name in active_tables:
    geojson_path = data_dir / (GRID_GEOJSON[area] if geom_type == "grids" else HEX_GEOJSON[area])

    if not FORCE_RELOAD_GEOMETRY and table_exists(table_name):
        print(f"  {table_name}: already in PostGIS — skipping")
        continue

    if not geojson_path.exists():
        print(f"  {table_name}: source file not found: {geojson_path.name} — skipping")
        continue

    print(f"  {table_name}: loading {geojson_path.name} ...")
    gdf = gpd.read_file(geojson_path).to_crs(32636)
    gdf.to_postgis(table_name, engine, if_exists="replace", index=False)
    print(f"  {table_name}: {len(gdf):,} rows uploaded")

In [ ]:
# Centroid columns and spatial indexes for all active tables
for _geom_type, _area, table_name in active_tables:
    ensure_centroid_column(table_name)
    ensure_spatial_index(table_name)
    print(f"  {table_name}: centroid + indexes ok")

## Section 4 — Planar Distance Calculations

Loops over `DISTANCE_SOURCES`.  Each column is computed with `ST_Distance` (planar,
EPSG:32636) using a KNN (`<->`) lateral lookup for speed.  Existing non-NULL values
are never overwritten.

**To add a new distance variable:** update `DISTANCE_SOURCES` at the top of the
notebook, then re-run only the cell below.

In [ ]:
for col, feat_table, applies_to in DISTANCE_SOURCES:

    # Determine which geometry tables this source applies to
    target_tables = []
    if applies_to in ("both", "grids") and PROCESS_GRIDS:
        target_tables += [GRID_TABLES[a] for a in AREAS]
    if applies_to in ("both", "hexbins") and PROCESS_HEXBINS:
        target_tables += [HEX_TABLES[a] for a in AREAS]

    if not target_tables:
        continue

    print(f"\n{col}  <-  {feat_table}")
    for table_name in target_tables:
        run_sql(f"""
            ALTER TABLE {table_name}
                ADD COLUMN IF NOT EXISTS {col} double precision;

            UPDATE {table_name} h
            SET {col} = (
                SELECT ST_Distance(h.centroid, f.geometry)
                FROM {feat_table} f
                ORDER BY h.centroid <-> f.geometry
                LIMIT 1
            )
            WHERE {col} IS NULL;
        """)
        print(f"  {table_name}: done")

print("\nPlanar distance calculations complete.")

## Section 5 — Geodesic Distances (Grids only)

For every entry in `DISTANCE_SOURCES` that applies to grids, computes a
`_geodesic` counterpart using `ST_DistanceSphere` and an absolute
`_diff` column.  Hexbins and already-populated columns are skipped.

Set `COMPUTE_GEODESIC = False` at the top to skip this section entirely.

In [ ]:
if not COMPUTE_GEODESIC or not PROCESS_GRIDS:
    print("Geodesic section skipped (COMPUTE_GEODESIC=False or no grids active).")
else:
    grid_only_tables = [GRID_TABLES[a] for a in AREAS]

    # Geodesic sources = all DISTANCE_SOURCES that touch grids
    geodesic_sources = [
        (col, feat_table)
        for col, feat_table, applies_to in DISTANCE_SOURCES
        if applies_to in ("both", "grids")
    ]

    # ----- Geodesic distance columns -----
    for col, feat_table in geodesic_sources:
        geo_col = f"{col}_geodesic"
        print(f"\n{geo_col}  <-  {feat_table}")
        for table_name in grid_only_tables:
            run_sql(f"""
                ALTER TABLE {table_name}
                    ADD COLUMN IF NOT EXISTS {geo_col} double precision;

                UPDATE {table_name} AS h
                SET {geo_col} = (
                    SELECT ST_DistanceSphere(
                        ST_Transform(h.centroid, 4326),
                        ST_Transform(f.geometry, 4326)
                    )
                    FROM {feat_table} AS f
                    ORDER BY h.centroid <-> f.geometry
                    LIMIT 1
                )
                WHERE {geo_col} IS NULL;
            """)
            print(f"  {table_name}: done")

    # ----- Planar / geodesic diff columns -----
    print("\nComputing diff columns ...")
    for col, _feat_table in geodesic_sources:
        geo_col  = f"{col}_geodesic"
        diff_col = f"{col}_diff"
        for table_name in grid_only_tables:
            run_sql(f"""
                ALTER TABLE {table_name}
                    ADD COLUMN IF NOT EXISTS {diff_col} double precision;

                UPDATE {table_name}
                SET {diff_col} = ABS({col} - {geo_col})
                WHERE {diff_col} IS NULL
                  AND {col}     IS NOT NULL
                  AND {geo_col} IS NOT NULL;
            """)
        print(f"  {diff_col}: done")

    print("\nGeodesic section complete.")

## Section 6 — Settlement Distances

Two-pass approach applied to **all** active tables:

- **Pass 1** — cells that intersect a settlement: `dist_to_settlement_boundary`
  (distance to the boundary of the largest-overlap settlement) and
  `dist_to_settlement_centroid`. `closest_settlement` is left NULL — those cells
  are already attributed via the `name` column.
- **Pass 2** — cells outside every settlement: `dist_to_settlement_boundary`
  = distance to the nearest settlement polygon edge;
  `closest_settlement` = name of that settlement.

All three columns use `ADD COLUMN IF NOT EXISTS` + `WHERE IS NULL` guards,
so re-running this section is always safe.

In [ ]:
for _geom_type, _area, table_name in active_tables:

    # Add columns (safe to call multiple times)
    run_sql(f"""
        ALTER TABLE {table_name}
            ADD COLUMN IF NOT EXISTS dist_to_settlement_boundary double precision;
        ALTER TABLE {table_name}
            ADD COLUMN IF NOT EXISTS dist_to_settlement_centroid double precision;
        ALTER TABLE {table_name}
            ADD COLUMN IF NOT EXISTS closest_settlement text;
    """)

    # Pass 1: cells that overlap a settlement — use the largest-overlap settlement
    run_sql(f"""
        UPDATE {table_name} c
        SET
            dist_to_settlement_boundary = sub.dist_boundary,
            dist_to_settlement_centroid = sub.dist_centroid
        FROM (
            SELECT DISTINCT ON (c2."OID")
                c2."OID",
                ST_Distance(
                    c2.centroid,
                    ST_Boundary(s.geometry)
                ) AS dist_boundary,
                ST_Distance(
                    c2.centroid,
                    ST_Centroid(s.geometry)
                ) AS dist_centroid
            FROM {table_name} c2
            JOIN settlements s
              ON ST_Intersects(c2.geometry, s.geometry)
            ORDER BY
                c2."OID",
                ST_Area(ST_Intersection(c2.geometry, s.geometry)) DESC
        ) sub
        WHERE c."OID" = sub."OID"
          AND c.dist_to_settlement_boundary IS NULL;
    """)

    # Pass 2: cells outside every settlement — nearest settlement via KNN
    run_sql(f"""
        UPDATE {table_name} c
        SET
            dist_to_settlement_boundary = near.dist_boundary,
            closest_settlement          = near.settlement_name
        FROM (
            SELECT
                c2."OID",
                ST_Distance(c2.centroid, s.geometry) AS dist_boundary,
                s.name                               AS settlement_name
            FROM {table_name} c2
            CROSS JOIN LATERAL (
                SELECT name, geometry
                FROM settlements
                ORDER BY c2.centroid <-> geometry
                LIMIT 1
            ) s
            WHERE c2.dist_to_settlement_boundary IS NULL
        ) near
        WHERE c."OID" = near."OID";
    """)

    print(f"  {table_name}: settlement distances done")

print("\nSettlement distance section complete.")

## Section 7 — Export

Reads each processed table from PostGIS and writes it to a GeoJSON file in
`data/processed`.  Set `EXPORT_RESULTS = False` at the top to skip.

In [ ]:
if not EXPORT_RESULTS:
    print("EXPORT_RESULTS=False — skipping export.")
else:
    # Output filename pattern per geometry type
    GRID_EXPORT_FNAME = {a: f"uganda_grids_{a}k_lcluc_10k_buffer_processed_v3.geojson"  for a in AREAS}
    HEX_EXPORT_FNAME  = {a: f"uganda_hexbins_{a}k_lcluc_10k_buffer_processed_v2.geojson" for a in AREAS}

    for geom_type, area, table_name in active_tables:
        fname    = GRID_EXPORT_FNAME[area] if geom_type == "grids" else HEX_EXPORT_FNAME[area]
        out_path = data_dir / fname
        print(f"  {table_name} -> {fname} ...")
        gdf = gpd.read_postgis(
            f"SELECT * FROM {table_name}",
            con=engine,
            geom_col="geometry",
        )
        gdf.to_file(out_path, driver="GeoJSON")
        print(f"  {table_name}: {len(gdf):,} rows written")

    print("\nExport complete.")